# Block T trên Kaggle: teacher 32B → student 4B/8B/14B

Bật **GPU T4 x2** và **Internet**. Vào **Add-ons > Secrets**, tạo `HF_TOKEN` (token có quyền ghi).
Mỗi tài khoản Kaggle mở notebook này, chỉ sửa ô cấu hình rồi Run All.

**Code nằm sẵn trong notebook** (ô thứ 4, sinh bởi `scripts/build_teach_notebook.py`). Hub chỉ chứa dữ liệu:
`inputs/` (chunks, seeds, index) và kết quả `teach/`; adapter student nằm ở repo model.
Khi code đổi: build lại notebook trên laptop rồi upload lại, không cần `teach.py push`.

| Bước | `STAGE` | Ghi chú |
| --- | --- | --- |
| T1 | `sample` | chọn hạt giống, đẩy `seeds.jsonl` lên `inputs/` |
| T2 | `generate` | mỗi tài khoản một `SHARD`; teacher viết query |
| T3 | `mine` | tìm negative khó từ index trên `inputs/index/`; đẩy `groups.jsonl` lên `teach/` |
| T4 | `judge` | mỗi tài khoản một `SHARD`; teacher chấm cặp |
| T5 | `train` | 2 student song song, mỗi T4 một student |

Session bị dừng giữa chừng thì Run All lại với cùng cấu hình: kết quả đã lưu trên Hub được tải về và chạy tiếp.


In [ ]:
STAGE = 'generate'          # 'sample' | 'generate' | 'mine' | 'judge' | 'train'
SHARD = 0                   # tài khoản thứ mấy: 0..SHARDS-1 (generate/judge)
SHARDS = 4                  # phải khớp mọi bước: sample sinh đúng SHARDS, generate/judge/mine đọc lại
SAMPLE_TARGET = 30000       # chỉ dùng ở STAGE='sample'
HUB_REPO = 'AncientWings/AncientWings_R2AI_STAGE3_ds'       # dataset có sẵn: code, input, kết quả teacher dưới teach/
HUB_MODEL_REPO = 'AncientWings/AncientWings_R2AI_STAGE3_model' # model repo có sẵn: adapter student dưới students/
TEACHER = 'qwen3-32b'       # vllm xung đột CUDA với torchaudio trên image Kaggle hiện tại -> dùng bnb 4-bit
BACKEND = 'hf'               # đổi về 'vllm'/None nếu Kaggle sửa image và vllm chạy được
HOURS = 11.0                # dừng sạch trước giới hạn 12 giờ
# T5: tối đa 2 job, mỗi job một GPU. (task, size): task in reranker|embedding|translator
TRAIN_JOBS = [('reranker', '4b'), ('embedding', '4b')]
RESUME = True
TRAIN_LIMIT = None          # vd. 2000 để chạy thử nhanh

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
if STAGE in ('generate', 'judge') and (BACKEND or 'vllm') == 'vllm':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'bitsandbytes',
                    'accelerate'], check=True)
import torch
print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Sinh tự động bởi scripts/build_teach_notebook.py: code của repo, nhúng trong notebook.
# Không sửa tay ô này; sửa code rồi chạy lại script đó.
import base64, hashlib, io, shutil, zipfile
from pathlib import Path
CODE_SHA256 = 'b100c95ee01a96d844e39023ddcfc3560397d4f806673e07685110a7fed1e135'
CODE_ZIP = (
    '...'
)
payload = base64.b64decode(CODE_ZIP)
if hashlib.sha256(payload).hexdigest() != CODE_SHA256:
    raise ValueError('Code nhúng bị hỏng; build lại notebook.')
CODE_DIR = Path('/kaggle/working/vibiomir/code')
shutil.rmtree(CODE_DIR, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(payload)).extractall(CODE_DIR)
print('Code:', CODE_SHA256[:12], sum(1 for _ in CODE_DIR.rglob('*.py')), 'file .py')


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

BASE = Path('/kaggle/working/vibiomir')
# Code is embedded in the cell above; the Hub only provides data.
# mine needs every 'generated' shard written so far; judge needs groups.jsonl;
# train needs the full teach/ tree (generated, judged, groups).
needs_teach = {'mine': ['teach/generated/**', 'teach/groups.jsonl'],
              'judge': ['teach/groups.jsonl'], 'train': ['teach/**']}
patterns = ['inputs/**'] + needs_teach.get(STAGE, [])
snapshot_download(HUB_REPO, repo_type='dataset', local_dir=BASE, allow_patterns=patterns,
                  token=os.environ['HF_TOKEN'])
TEACH = BASE / 'teach'
TEACH.mkdir(exist_ok=True)
print(sorted(str(p.relative_to(BASE)) for p in BASE.rglob('*')
             if p.is_file() and 'code' not in p.parts and '.cache' not in p.parts))

In [ ]:
SCRIPT = str(BASE / 'code' / 'scripts' / 'teach.py')
INPUTS = BASE / 'inputs'

def launch(args, gpu=None):
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.Popen([sys.executable, '-u', SCRIPT, *map(str, args)], env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

def follow(processes):
    import selectors
    selector = selectors.DefaultSelector()
    for label, process in processes:
        selector.register(process.stdout, selectors.EVENT_READ, label)
    open_streams = len(processes)
    while open_streams:
        for key, _ in selector.select():
            line = key.fileobj.readline()
            if not line:
                selector.unregister(key.fileobj)
                open_streams -= 1
                continue
            print(f'[{key.data}] {line}', end='', flush=True)
    codes = {label: process.wait() for label, process in processes}
    print(codes)
    if any(codes.values()):
        raise RuntimeError(f'Có tiến trình lỗi: {codes}')

common = ['--out', TEACH, '--shard', SHARD, '--shards', SHARDS, '--teacher', TEACHER,
          '--hub-repo', HUB_REPO, '--hours', HOURS] + (['--backend', BACKEND] if BACKEND else [])
if STAGE == 'sample':
    follow([('sample', launch(['sample', '--chunks', INPUTS / 'chunks.parquet', '--out', TEACH,
                                '--target', SAMPLE_TARGET, '--shards', SHARDS,
                                '--hub-repo', HUB_REPO]))])
elif STAGE == 'mine':
    index_root = INPUTS / 'index'
    index_dirs = sorted(p for p in index_root.iterdir() if p.is_dir()) if index_root.is_dir() else []
    if not index_dirs:
        raise FileNotFoundError(f'Không có index nào dưới {index_root}; '
                                 'đẩy work/<run>/index từ laptop bằng teach.py push trước.')
    follow([('mine', launch(['mine', '--teach', TEACH, '--chunks', INPUTS / 'chunks.parquet',
                              '--index', index_dirs[0], '--extra-index', *index_dirs[1:],
                              '--shards', SHARDS, '--device', 'cuda', '--search-device', 'cuda',
                              '--hub-repo', HUB_REPO]))])
elif STAGE == 'generate':
    follow([('generate', launch(['generate', '--seeds', INPUTS / 'seeds.jsonl', *common]))])
elif STAGE == 'judge':
    follow([('judge', launch(['judge', '--groups', TEACH / 'groups.jsonl',
                               '--chunks', INPUTS / 'chunks.parquet', *common]))])
elif STAGE == 'train':
    if len(TRAIN_JOBS) > torch.cuda.device_count():
        raise ValueError('Mỗi job cần một GPU riêng.')
    jobs = []
    for gpu, (task, size) in enumerate(TRAIN_JOBS):
        args = ['train', '--task', task, '--size', size, '--teach', TEACH,
                '--chunks', INPUTS / 'chunks.parquet', '--out', '/kaggle/working/students',
                '--hub-repo', HUB_MODEL_REPO, '--hours', HOURS]
        if RESUME:
            args.append('--resume')
        if TRAIN_LIMIT:
            args += ['--limit', TRAIN_LIMIT]
        jobs.append((f'{task}-{size}', launch(args, gpu=gpu)))
    follow(jobs)